# SmolVLA few-shot adaptation on LIBERO

Notebook выполняет весь pipeline сверху вниз. Checkpoints, evaluation и метрики сохраняются на диск.

## 1. Setup

In [1]:
from pathlib import Path
import math
import os
import sys
import warnings

warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["HF_HUB_VERBOSITY"] = "error"
os.environ["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

if sys.version_info[:2] != (3, 12):
    raise RuntimeError(
        f"Python 3.12 is required, got {sys.version.split()[0]}. "
        "Recreate the server environment before installing dependencies."
    )

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    raise FileNotFoundError("Run the notebook from the repository root.")

In [2]:
import shutil
import subprocess

import pandas as pd
import torch
import lerobot

from src import analysis
from src import bonus
from src import data
from src import evaluation
from src import h1_dynamics
from src import h2_progress
from src import train_baselines
from src.settings import BUDGETS, RESULTS, TARGETS

subprocess.run([sys.executable, "-m", "pip", "check"], check=True)

if lerobot.__version__ != "0.6.1":
    raise RuntimeError(f"Expected LeRobot 0.6.1, got {lerobot.__version__}")
if not torch.__version__.startswith("2.11.0"):
    raise RuntimeError(f"Expected PyTorch 2.11.0, got {torch.__version__}")
if torch.version.cuda != "12.8":
    raise RuntimeError(f"Expected CUDA 12.8 PyTorch build, got {torch.version.cuda}")
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available")
if shutil.which("git") is None:
    raise RuntimeError("git is required")
if shutil.which("ffmpeg") is None:
    raise RuntimeError("ffmpeg is required")

device = torch.cuda.get_device_properties(0)
print(
    f"Python {sys.version.split()[0]} | LeRobot {lerobot.__version__} | "
    f"PyTorch {torch.__version__} | CUDA {torch.version.cuda}"
)
print(
    f"GPU: {device.name} | VRAM: {device.total_memory / 2**30:.1f} GB | "
)
if device.total_memory < 80 * 2**30:
    raise RuntimeError(
        "This fixed-batch configuration assumes an >=80 GB GPU. "
        f"Detected {device.total_memory / 2**30:.1f} GB."
    )

SEEN_BATCH = 32
TARGET_BATCH = 8
H1_PRIOR_BATCH = 32
TIME_REWARDER_BATCH = 16

AUX_SEED = 7
EVAL_EPISODES = 20
EVAL_SEED = 10_000
TIME_REWARDER_EPOCHS = 20
RABC_KAPPA = 0.01

pd.set_option("display.max_columns", None)
RESULTS.mkdir(parents=True, exist_ok=True)

No broken requirements found.
Python 3.12.3 | LeRobot 0.6.1 | PyTorch 2.11.0+cu128 | CUDA 12.8
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition | VRAM: 95.0 GB | 


### External environments

До скачивания данных проверяются отдельные Blackwell-compatible environments для TimeRewarder и Robometer.

In [3]:
timerewarder_repo = h2_progress.setup_timerewarder()
robometer_repo = bonus.setup_robometer()
print("TimeRewarder:", timerewarder_repo)
print("Robometer:", robometer_repo)

$ git log -1 --format=TimeRewarder:%H
TimeRewarder:f54234b67bd3f1fa190f62498d38513a2140f23f
$ uv python install 3.9
$ uv venv --python 3.9 _external/TimeRewarder/.venv
$ uv pip install --python _external/TimeRewarder/.venv/bin/python torch==2.7.1+cu128 torchvision==0.22.1+cu128 --index-url https://download.pytorch.org/whl/cu128


Python 3.9 is already installed
Using CPython 3.9.25
Creating virtual environment at: _external/TimeRewarder/.venv
Activate with: source _external/TimeRewarder/.venv/bin/activate
Using Python 3.9.25 environment at: _external/TimeRewarder/.venv
Resolved 28 packages in 344ms
Installed 28 packages in 185ms
 + filelock==3.19.1
 + fsspec==2025.10.0
 + jinja2==3.1.6
 + markupsafe==3.0.2
 + mpmath==1.3.0
 + networkx==3.2.1
 + numpy==2.0.2
 + nvidia-cublas-cu12==12.8.3.14
 + nvidia-cuda-cupti-cu12==12.8.57
 + nvidia-cuda-nvrtc-cu12==12.8.61
 + nvidia-cuda-runtime-cu12==12.8.57
 + nvidia-cudnn-cu12==9.7.1.26
 + nvidia-cufft-cu12==11.3.3.41
 + nvidia-cufile-cu12==1.13.0.11
 + nvidia-curand-cu12==10.3.9.55
 + nvidia-cusolver-cu12==11.7.2.55
 + nvidia-cusparse-cu12==12.5.7.53
 + nvidia-cusparselt-cu12==0.6.3
 + nvidia-nccl-cu12==2.26.2
 + nvidia-nvjitlink-cu12==12.8.61
 + nvidia-nvtx-cu12==12.8.55
 + pillow==11.3.0
 + setuptools==78.1.0
 + sympy==1.14.0
 + torch==2.7.1+cu128
 + torchvision==0.22.1

$ uv pip install --python _external/TimeRewarder/.venv/bin/python setuptools==80.9.0 wheel
$ env MMCV_WITH_OPS=0 uv pip install --python _external/TimeRewarder/.venv/bin/python --no-build-isolation mmcv==1.7.1


Using Python 3.9.25 environment at: _external/TimeRewarder/.venv
Resolved 3 packages in 123ms
Uninstalled 1 package in 8ms
Installed 3 packages in 13ms
 + packaging==26.3
 - setuptools==78.1.0
 + setuptools==80.9.0
 + wheel==0.48.0
Using Python 3.9.25 environment at: _external/TimeRewarder/.venv
Resolved 10 packages in 157ms
Installed 7 packages in 16ms
 + addict==2.4.0
 + mmcv==1.7.1
 + opencv-python==5.0.0.93
 + platformdirs==4.4.0
 + pyyaml==6.0.3
 + tomli==2.4.1
 + yapf==0.43.0
Using Python 3.9.25 environment at: _external/TimeRewarder/.venv


$ uv pip install --python _external/TimeRewarder/.venv/bin/python numpy==1.24.4 einops==0.8.0 timm==1.0.9 ftfy==6.2.3 regex==2024.9.11 decord==0.6.0 opencv-python==4.5.3.56 yacs==0.1.8 tensorboard==2.14.0 scipy==1.10.1 imageio==2.9.0 imageio-ffmpeg==0.4.4 pandas==1.3.0 matplotlib==3.4.2 termcolor==1.1.0 protobuf==3.20.1 huggingface_hub==0.25.1


Resolved 77 packages in 262ms
Uninstalled 2 packages in 15ms
Installed 47 packages in 61ms
 + absl-py==2.3.1
 + certifi==2026.7.22
 + cffi==2.0.0
 + charset-normalizer==3.5.1
 + cryptography==50.0.0
 + cycler==0.12.1
 + decord==0.6.0
 + einops==0.8.0
 + ftfy==6.2.3
 + google-auth==2.50.0
 + google-auth-oauthlib==1.0.0
 + grpcio==1.80.0
 + huggingface-hub==0.25.1
 + idna==3.19
 + imageio==2.9.0
 + imageio-ffmpeg==0.4.4
 + importlib-metadata==8.7.1
 + kiwisolver==1.4.7
 + markdown==3.9
 + matplotlib==3.4.2
 - numpy==2.0.2
 + numpy==1.24.4
 + oauthlib==3.3.1
 - opencv-python==5.0.0.93
 + opencv-python==4.5.3.56
 + pandas==1.3.0
 + protobuf==3.20.1
 + pyasn1==0.6.4
 + pyasn1-modules==0.4.2
 + pycparser==2.23
 + pyparsing==3.3.2
 + python-dateutil==2.9.0.post0
 + pytz==2026.3.post1
 + regex==2024.9.11
 + requests==2.32.5
 + requests-oauthlib==2.0.0
 + safetensors==0.7.0
 + scipy==1.10.1
 + six==1.17.0
 + tensorboard==2.14.0
 + tensorboard-data-server==0.7.2
 + termcolor==1.1.0
 + timm==1.0.

$ uv pip check --python _external/TimeRewarder/.venv/bin/python


CalledProcessError: Command '['uv', 'pip', 'check', '--python', '_external/TimeRewarder/.venv/bin/python']' returned non-zero exit status 1.

## 2. Data

Подготавливается полный `libero_90`, исправляется gripper convention и создаются физические first-K subsets для трёх target tasks.

In [ ]:
data_info = data.prepare_data()
data_info

In [ ]:
seen_steps = math.ceil(
    2.0 * data_info["seen_train_frames"] / SEEN_BATCH
)
h1_prior_steps = math.ceil(
    1.0 * data_info["seen_train_frames"] / H1_PRIOR_BATCH
)

target_steps = {}
for task_id in TARGETS:
    for k in BUDGETS:
        frames = data_info["target_frames"][f"t{task_id}_k{k}"]
        samples = max(6400, math.ceil(2.5 * frames))
        target_steps[(task_id, k)] = math.ceil(samples / TARGET_BATCH)

schedule = pd.DataFrame(
    [
        {
            "task_id": task_id,
            "K": k,
            "frames": data_info["target_frames"][f"t{task_id}_k{k}"],
            "steps": target_steps[(task_id, k)],
        }
        for task_id in TARGETS
        for k in BUDGETS
    ]
)

print("seen steps:", seen_steps)
print("H1 prior steps:", h1_prior_steps)
display(schedule)

## 3. Replay check

Перед training одна исправленная demonstration каждой target task должна успешно replay-иться в LIBERO.

In [ ]:
smoke = pd.DataFrame(
    [
        data.replay_gripper_smoke(task_id, k=5, episode=0)
        for task_id in TARGETS
    ]
)
smoke.to_csv(RESULTS / "replay_smoke.csv", index=False)
display(smoke)

## 4. Seen policy

`lerobot/smolvla_base` дообучается на полном исправленном `libero_90`.

In [ ]:
seen = train_baselines.train_seen(
    steps=seen_steps,
    batch_size=SEEN_BATCH,
    seed=AUX_SEED,
)
print(seen)

## 5. K=0 and wrong instruction

Seen checkpoint оценивается zero-shot. Затем те же simulator seeds повторяются с instruction другой target task.

In [ ]:
zero_json = evaluation.eval_checkpoint(
    seen,
    "zero_shot",
    tuple(TARGETS),
    n_episodes=EVAL_EPISODES,
    seed=EVAL_SEED,
)
zero_raw = analysis.collect_eval_results(
    [
        {
            "eval_json": str(zero_json),
            "method": "zero_shot",
            "budget": 0,
            "train_seed": None,
        }
    ]
)

wrong = evaluation.eval_wrong_language(
    seen,
    n_episodes=EVAL_EPISODES,
    seed=EVAL_SEED,
)
language = analysis.language_control(
    zero_raw,
    wrong,
    start_seed=EVAL_SEED,
)
language_summary = analysis.language_control_summary(language)

language.to_csv(RESULTS / "language_control.csv", index=False)
language_summary.to_csv(
    RESULTS / "language_control_summary.csv",
    index=False,
)
display(language_summary)

## 6. Baselines

Обучаются обязательный native fine-tune и matched LoRA-only control на `K=5/10/25`, seeds `42/123`.

In [ ]:
native_checkpoints = train_baselines.train_native_grid(
    seen,
    target_steps,
    batch_size=TARGET_BATCH,
)
lora_checkpoints = train_baselines.train_lora_grid(
    seen,
    target_steps,
    batch_size=TARGET_BATCH,
)

print("native:", len(native_checkpoints))
print("LoRA:", len(lora_checkpoints))

## 7. H1: latent dynamics

Seen-only dynamics prior обучается один раз и затем используется как frozen regularizer для target LoRA adaptation.

In [ ]:
h1_prior = h1_dynamics.train_h1_prior(
    seen,
    steps=h1_prior_steps,
    batch_size=H1_PRIOR_BATCH,
    seed=AUX_SEED,
    horizon=10,
)

h1_checkpoints = h1_dynamics.train_h1_grid(
    seen,
    h1_prior,
    target_steps,
    batch_size=TARGET_BATCH,
    lambda_dyn=0.1,
)

print("H1:", len(h1_checkpoints))

## 8. H2: video progress

Для каждого `task x K` TimeRewarder обучается по passive target video. Его progress используется official RA-BC для target LoRA adaptation.

In [ ]:
reward_models, progress_files = h2_progress.prepare_h2_rewards(
    seed=AUX_SEED,
    epochs=TIME_REWARDER_EPOCHS,
    batch_size=TIME_REWARDER_BATCH,
)

h2_checkpoints = h2_progress.train_h2_grid(
    seen,
    progress_files,
    target_steps,
    batch_size=TARGET_BATCH,
    kappa=RABC_KAPPA,
)

print("TimeRewarder:", len(reward_models))
print("H2:", len(h2_checkpoints))

## 9. Main evaluation

Каждый adapted checkpoint оценивается только на своей target task. Для каждой cell используются 20 episodes.

In [ ]:
eval_specs = [
    {
        "eval_json": str(zero_json),
        "method": "zero_shot",
        "budget": 0,
        "train_seed": None,
    }
]

for method, checkpoints in [
    ("native", native_checkpoints),
    ("LoRA", lora_checkpoints),
    ("H1", h1_checkpoints),
    ("H2", h2_checkpoints),
]:
    eval_specs.extend(
        evaluation.evaluate_grid(
            checkpoints,
            method,
            n_episodes=EVAL_EPISODES,
            seed=EVAL_SEED,
        )
    )

raw = analysis.collect_eval_results(eval_specs)
raw.to_csv(RESULTS / "evaluation.csv", index=False)

curve_raw = analysis.attach_shared_zero_shot(
    raw,
    methods=("native", "LoRA", "H1", "H2"),
)
success = analysis.success_rates(curve_raw)
curve = analysis.cost_curve_summary(curve_raw)

success.to_csv(RESULTS / "success_rates.csv", index=False)
curve.to_csv(RESULTS / "cost_curve.csv", index=False)
analysis.plot_cost_curve(
    curve_raw,
    RESULTS / "cost_curve.png",
    methods=("native", "LoRA", "H1", "H2"),
)

display(success[["method", "budget", "train_seed", "task_id", "success_rate", "ci_low", "ci_high"]])
display(curve)

## 10. Bonus B

Для `K=5, seed=42` одни и те же сохранённые rollout videos оцениваются TimeRewarder и Robometer. True checkpoint success берётся по всем 20 simulator episodes.

In [ ]:
pool = bonus.make_video_pool(
    raw,
    k=5,
    train_seed=42,
    max_videos_per_checkpoint=10,
)

timerewarder_scores = bonus.score_timerewarder(
    pool,
    reward_models,
    k=5,
)
robometer_scores = bonus.score_robometer(pool)

bonus_scores = pd.concat(
    [timerewarder_scores, robometer_scores],
    ignore_index=True,
)

bonus_dir = RESULTS / "bonus_b"
bonus_dir.mkdir(parents=True, exist_ok=True)
bonus_scores.to_csv(bonus_dir / "scores.csv", index=False)

In [ ]:
true_success = bonus.checkpoint_success(
    raw,
    k=5,
    train_seed=42,
)
checkpoint_scores, ranking = bonus.rank_checkpoints(
    bonus_scores,
    true_success,
)
pressure = bonus.reward_pressure(checkpoint_scores)

checkpoint_scores.to_csv(
    bonus_dir / "checkpoint_scores.csv",
    index=False,
)
ranking.to_csv(bonus_dir / "ranking.csv", index=False)
pressure.to_csv(
    bonus_dir / "reward_pressure.csv",
    index=False,
)

display(ranking)
display(pressure)

## 11. Failure analysis

Сохраняются реальные failed rollout videos. Для отчёта выбираются три характерных случая и формулируются hypothesis, alternative и separating experiment.

In [ ]:
failures = analysis.failure_candidates(raw)
failures.to_csv(
    RESULTS / "failure_candidates.csv",
    index=False,
)
display(failures.head(30))

## 12. Result files

Основные результаты находятся в `results/`; model checkpoints, raw LeRobot eval artifacts и run logs — в `outputs/`.